In [4]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import pandas as pd
import numpy as np
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime, timedelta
# 環境変数を読み込むためのライブラリをインポート
from dotenv import load_dotenv

# 1. 同じフォルダにある .env ファイルから設定を読み込む
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

# .envファイルのパスを指定して読み込み
dotenv_path = os.path.join(current_dir, '.env')
load_dotenv(dotenv_path)

# 環境変数からスプレッドシートIDを取得（コード内には直接書かない）
SPREADSHEET_ID = os.getenv('GOOGLE_SPREADSHEET_ID')

if not SPREADSHEET_ID:
    raise ValueError("❌ .env ファイルに GOOGLE_SPREADSHEET_ID が設定されていません。")


# ==========================================
# 2. データの疑似生成（これまでと同じ）
# ==========================================
np.random.seed(42)
dates = [datetime(2026, 1, 1) + timedelta(days=i) for i in range(100)]
cities = ['Tokyo', 'New York', 'London', 'Paris']

data = {
    'Date': np.random.choice(dates, 500),
    'City': np.random.choice(cities, 500),
    'Sales': np.random.randint(100, 1500, 500),
    'Profit_Rate': np.random.uniform(0.1, 0.4, 500)
}
df = pd.DataFrame(data)
df['Profit'] = (df['Sales'] * df['Profit_Rate']).astype(int)
df['Profit_Rate'] = df['Profit_Rate'].round(2)
df = df.sort_values(by='Date').reset_index(drop=True)
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')


# ==========================================
# 3. Googleスプレッドシートへの安全な接続
# ==========================================
# 環境変数から「鍵のファイル名」を取得
key_file_name = os.getenv('KEY_PATH')

if not key_file_name:
    raise ValueError("❌ .env ファイルに GOOGLE_KEY_FILE_NAME が設定されていません。")

# 実際のファイル名を見せないように結合
key_path = os.path.join(current_dir, key_file_name)

# 鍵ファイル（secret_key.json）のパス
scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
creds = ServiceAccountCredentials.from_json_keyfile_name(key_path, scope)
client = gspread.authorize(creds)

try:
    # 環境変数から読み込んだIDを使ってスプレッドシートを開く
    sh = client.open_by_key(SPREADSHEET_ID)
    worksheet = sh.get_worksheet(0)
    
    # シートのクリアとデータの一括書き込み
    worksheet.clear()
    data_to_write = [df.columns.values.tolist()] + df.values.tolist()
    worksheet.update(values=data_to_write, range_name='A1')
    
    print("=" * 60)
    print("🎉 【セキュリティ版】スプレッドシートへの直接書き込みに成功しました！")
    print("=" * 60)
    print("重要な情報はすべて外部ファイル（.env）に隠されています。")
    print("これで安心してコードを管理できます。さあ、Looker Studioを開きましょう！")
    print("=" * 60)

except Exception as e:
    print(f"❌ エラーが発生しました。設定を見直してください:\n{e}")

🎉 【セキュリティ版】スプレッドシートへの直接書き込みに成功しました！
重要な情報はすべて外部ファイル（.env）に隠されています。
これで安心してコードを管理できます。さあ、Looker Studioを開きましょう！
